# 04_5 · Transformer jerárquico multitarea

Este cuaderno prueba una alternativa jerárquica **conjunta**: un único encoder alimenta una cabeza binaria `SEGURO/daño` y cinco cabezas para las categorías gruesas de daño. Se compara contra el mejor Transformer plano registrado por `04_2`, usando exactamente las mismas particiones.

La hipótesis es que la supervisión binaria compartida puede mejorar la detección global sin la propagación rígida de errores de una cascada. El puntaje de cada categoría es `p(daño) × p(categoría | representación)`, con una penalización suave de consistencia cuando una categoría supera a la probabilidad global de daño.

**Requisito:** ejecutar en `04_2` las evaluaciones de los Transformers y su comparación final.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'scripts_auxiliares' / 'experimentos_jerarquicos.py').exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto.')

ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Image, Markdown, display
from scripts_auxiliares import experimentos_jerarquicos as hj

print('Raíz:', ROOT)
print('PyTorch:', hj.torch.__version__)
print('Dispositivo:', hj._device())

## 1. Auditoría y referencia plana congelada

In [ ]:
context = hj.load_frozen_context()
audit = hj.context_summary(context)
display(audit)
print('Referencia plana:', audit.attrs['flat_reference'])
print('PR-AUC macro de validación usada para seleccionarla:',
      f"{audit.attrs['flat_selection_metric']:.4f}")
print('SHA-256 dataset:', context['dataset_sha256'])
print('SHA-256 manifiesto:', context['manifest_sha256'])

## 2. Metodología

El modelo comparte el encoder de la referencia plana y optimiza:

`L = 0,50·BCE(daño) + 1,00·BCE(categorías) + 0,10·consistencia jerárquica`.

Las ponderaciones positivas son la raíz de `negativos/positivos`, estrategia moderada que evita que la categoría minoritaria quede ignorada sin imponer el cociente completo. El mejor checkpoint se selecciona únicamente por PR-AUC macro de daño en validación y los cinco umbrales se calibran allí. Test queda reservado para una evaluación final.

La comparación usa PR-AUC macro, F1 macro, recall por categoría y falsos negativos. Los intervalos percentiles 95 % se obtienen mediante bootstrap pareado de videos completos. La regla de mejora exige IC de Δ PR-AUC completamente positivo y ausencia de aumento compatible de la tasa de falsos negativos.

Sólo se entrenan etiquetas gruesas; etiquetas finas y flags transversales no son objetivos ni entradas.

In [ ]:
# Configuración declarada antes de abrir el test.
FORCE = False
BOOTSTRAP_REPLICATES = 1_000
display({
    'peso_loss_binaria': hj.JOINT_BINARY_LOSS_WEIGHT,
    'peso_loss_categorias': hj.JOINT_CATEGORY_LOSS_WEIGHT,
    'peso_consistencia': hj.JOINT_CONSISTENCY_LOSS_WEIGHT,
    'epocas_maximas': hj.MAX_EPOCHS,
    'recall_objetivo_auto_seguro': hj.GATE_VALIDATION_RECALL_TARGET,
})

## 3. Fine-tuning conjunto con barra de avance

La celda guarda checkpoint, tokenizer, historial por época, probabilidades, métricas, gráficos, tablas comparativas e informe Markdown. Con `FORCE=False`, un resultado compatible ya terminado sólo se vuelve a cargar.

In [ ]:
result = hj.run_joint_experiment(
    force=FORCE,
    bootstrap_replicates=BOOTSTRAP_REPLICATES,
)
print('Experimento:', result['experiment_label'])
print('Mejor época:', result['training']['best_epoch'])
print('Resultado:', hj.result_path(result['experiment_key']))
print('Informe:', hj.report_path(result['experiment_key']))

## 4. Resultado global, por categoría e inferencia pareada

In [ ]:
tables = hj.load_experiment_tables(result['experiment_key'])
display(tables['comparison'].round(4))
display(tables['categories'].round(4))
display(tables['bootstrap'].round(4))

In [ ]:
figure_dir = hj.FIGURES_ROOT / result['experiment_key']
for name in ('comparacion_global_test.png',
             'recall_por_categoria_test.png',
             'bootstrap_deltas_test.png'):
    display(Image(filename=str(figure_dir / name)))

## 5. Capacidad de abstención y conclusión

In [ ]:
op = result['selective_operation']
display({
    'umbral_auto_seguro': op['low_auto_safe_threshold'],
    'umbral_auto_daño': op['high_auto_damage_threshold'],
    'tasa_revision_test': op['review_rate'],
    'cobertura_automatica_test': op['automatic_coverage'],
    'daños_auto_pasados_como_seguro':
        op['damage_automatic_safe_false_negatives'],
    'decision_estadistica': result['decision']['status'],
    'reemplazar_plano': result['decision']['replace_flat_model'],
})
display(Markdown(hj.report_path(result['experiment_key']).read_text(encoding='utf-8')))

## Referencias (APA 7)

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC.

Geifman, Y., & El-Yaniv, R. (2017). Selective classification for deep neural networks. *International Conference on Learning Representations*. https://openreview.net/forum?id=ryMhqj0ct7

Jiang, T., Wang, D., Sun, L., Yang, H., Zhao, Z., & Zhuang, F. (2022). Exploiting global and local hierarchies for hierarchical text classification. In *Proceedings of the 2022 Conference on Empirical Methods in Natural Language Processing* (pp. 4030–4039). Association for Computational Linguistics. https://aclanthology.org/2022.emnlp-main.268/

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

Zhou, J., Ma, C., Long, D., Xu, G., Ding, N., Zhang, H., Xie, P., & Liu, G. (2020). Hierarchy-aware global model for hierarchical text classification. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics* (pp. 1106–1117). Association for Computational Linguistics. https://doi.org/10.18653/v1/2020.acl-main.104